# Limpeza da base de dados

#### Essa etapa se destina ao tratamento dos problemas encontrados na fase de Exploração Inicial, visando preparar a base para a análise.

#### Os pontos que serão tratados nessa etapa são:

  - Reformatar os tipos das colunas
  - Avaliar o impacto dos valores nulos na análise da base de dados e avaliar a possibilidade de preenchê-los.
  - Definir um padrão de classificação indicativa para a variável `Certificate`.
  - Tratar o valor inconsistente na coluna `Release_Year` 

#### Além disso, a fim de otimizar a etapa de análise, nessa etapa serão criadas variáveis de interesse desse processo, assim como serão excluídas colunas que não acrescentam a ele.

---
## Configuração inicial

### Importação das bibliotecas

In [123]:
import pandas as pd
import numpy as np

### Carregamento da base de dados

In [124]:
raw_imdb_df = pd.read_csv("../data/raw/imdb_top_1000.csv")
clean_imdb_df = raw_imdb_df.copy()

---
## Padronização e reformatação das colunas

Para facilitar a manipulação das colunas do DataFrame, os nomes serão padronizados em letras minúsculas e snake case. Em seguida, os tipos de dados serão convertidos para o formato mais adequado, conforme definido na etapa de Exploração Inicial.


| Coluna | Descrição | Tipo atual | Tipo ideal |
|--------|-----------|------------|----------------|
| `Poster_Link` | URL do pôster do filme no IMDb. | `object` | `string` |
| `Series_Title` | Título do filme. | `object` | `string` |
| `Released_Year` | Ano de lançamento do filme. | `object` | `int64` |
| `Certificate` | Classificação indicativa do filme. | `object` | `category` |
| `Runtime` | Duração do filme em minutos. | `object` | `int64` |
| `Genre` | Gêneros do filme, separados por vírgulas. | `object` | `object` no formato `list[str]` |
| `IMDB_Rating` | Nota atribuída pelos usuários do IMDb (0–10). | `float64` | `float64` |
| `Overview` | Breve sinopse do filme. | `object` | `string` |
| `Meta_score` | Nota atribuída pelo Metacritic (0–100). | `float64` | `float64`|
| `Director` | Nome do diretor do filme. | `object` | `string` |
| `Star1` | Primeiro ator principal listado. | `object` | `string` |
| `Star2` | Segundo ator principal listado. | `object` | `string` |
| `Star3` | Terceiro ator principal listado. | `object` | `string` |
| `Star4` | Quarto ator principal listado. | `object` | `string` |
| `No_of_votes` | Número de avaliações recebidas no IMDb. | `int64` | `int64` |
| `Gross` | Receita bruta do filme (em dólares). | `object` | `float64` |

As colunas `released_year` e `certificate` apresentam inconsistências que impedem a conversão de tipos nessa etapa. Assim, elas serão convertidas após a correção dos problemas identificados.

### Nomes das colunas

In [125]:
clean_imdb_df.columns = (
    clean_imdb_df.columns
    .str.lower()
    .str.strip()
)
clean_imdb_df.columns

Index(['poster_link', 'series_title', 'released_year', 'certificate',
       'runtime', 'genre', 'imdb_rating', 'overview', 'meta_score', 'director',
       'star1', 'star2', 'star3', 'star4', 'no_of_votes', 'gross'],
      dtype='object')

### Colunas textuais

In [126]:
text_columns = [
    "poster_link",
    "series_title",
    "overview",
    "director",
    "star1",
    "star2",
    "star3",
    "star4",
]

clean_imdb_df[text_columns] = clean_imdb_df[text_columns].astype("string")

### Coluna de gêneros

In [127]:
clean_imdb_df["genre"] = clean_imdb_df["genre"].str.split(", ")
clean_imdb_df["genre"]

0                         [Drama]
1                  [Crime, Drama]
2          [Action, Crime, Drama]
3                  [Crime, Drama]
4                  [Crime, Drama]
                  ...            
995      [Comedy, Drama, Romance]
996              [Drama, Western]
997         [Drama, Romance, War]
998                  [Drama, War]
999    [Crime, Mystery, Thriller]
Name: genre, Length: 1000, dtype: object

### Tempo de duração

In [128]:
clean_imdb_df["runtime"] = (
    clean_imdb_df["runtime"]
    .str.replace(" min", "", regex=False)
    .astype(int)
)

clean_imdb_df["runtime"]

0      142
1      175
2      152
3      202
4       96
      ... 
995    115
996    201
997    118
998     97
999     86
Name: runtime, Length: 1000, dtype: int64

### Receita do filme

In [129]:
clean_imdb_df["gross"] = (
    clean_imdb_df["gross"]
    .str.replace(",", "", regex=False)
    .astype(float)
)

clean_imdb_df["gross"]

0       28341469.0
1      134966411.0
2      534858444.0
3       57300000.0
4        4360000.0
          ...     
995            NaN
996            NaN
997     30500000.0
998            NaN
999            NaN
Name: gross, Length: 1000, dtype: float64

---
## Valores inconsistentes

Correção das inconsistências encontradas na fase de Exploração Inicial da base de dados.

### Coluna de classificação indicativa

Foi identificado que a coluna `certificate` não seguia um único padrão nacional de classificação indicativa. Portanto, será adotado o padrão americano (G, PG, PG-13, R, NC-17 e Unrated). Além disso, a coluna será reformatada como o tipo `category`.

In [130]:
clean_imdb_df["certificate"].value_counts()

certificate
U           234
A           197
UA          175
R           146
PG-13        43
PG           37
Passed       34
G            12
Approved     11
TV-PG         3
GP            2
TV-14         1
Unrated       1
TV-MA         1
16            1
U/A           1
Name: count, dtype: int64

In [131]:
certificate_mapping = {
    "U": "G",
    "Approved": "G",
    "Passed": "G",
    "G": "G",

    "GP": "PG",
    "PG": "PG",
    "TV-PG": "PG",

    "UA": "PG-13",
    "U/A": "PG-13",
    "PG-13": "PG-13",
    "TV-14": "PG-13",

    "R": "R",
    "16": "R",

    "A": "NC-17",
    "TV-MA": "NC-17"
}

clean_imdb_df["certificate"] = (
    clean_imdb_df["certificate"]
    .replace(certificate_mapping)
)

clean_imdb_df["certificate"].value_counts()

certificate
G          291
PG-13      220
NC-17      198
R          147
PG          42
Unrated      1
Name: count, dtype: int64

In [132]:
clean_imdb_df["certificate"] = (
    clean_imdb_df["certificate"]
    .str.strip()
    .astype("category")
)
clean_imdb_df["certificate"].dtype

CategoricalDtype(categories=['G', 'NC-17', 'PG', 'PG-13', 'R', 'Unrated'], ordered=False, categories_dtype=object)

### Coluna de ano de lançamento

Durante a inspeção inicial da base, foi identificada uma entrada "PG" na coluna `released_year`. Como se trata de um único caso, será corrigido manualmente e a coluna será reformatada como `int`.

In [133]:
clean_imdb_df.loc[clean_imdb_df["released_year"] == "PG"]

,poster_link,series_title,released_year,certificate,runtime,genre,imdb_rating,overview,meta_score,director,star1,star2,star3,star4,no_of_votes,gross
966,https://m.media-amazon.com/images/M/MV5BNjEzYj...,Apollo 13,PG,G,140,"[Adventure, Drama, History]",7.6,NASA must devise a strategy to return Apollo 1...,77.0,Ron Howard,Tom Hanks,Bill Paxton,Kevin Bacon,Gary Sinise,269197,173837933.0


In [134]:
clean_imdb_df.loc[
    clean_imdb_df["released_year"] == "PG",
    "released_year"
] = 1995

clean_imdb_df.loc[clean_imdb_df["series_title"] == "Apollo 13"]



,poster_link,series_title,released_year,certificate,runtime,genre,imdb_rating,overview,meta_score,director,star1,star2,star3,star4,no_of_votes,gross
966,https://m.media-amazon.com/images/M/MV5BNjEzYj...,Apollo 13,1995,G,140,"[Adventure, Drama, History]",7.6,NASA must devise a strategy to return Apollo 1...,77.0,Ron Howard,Tom Hanks,Bill Paxton,Kevin Bacon,Gary Sinise,269197,173837933.0


In [135]:
clean_imdb_df["released_year"] = clean_imdb_df["released_year"].astype(int)

---
## Valores ausentes

Na etapa de Exploração Incial foi identificado que as colunas `certificate`, `meta_score` e `gross` apresentaram aproximadamente 10% de registros nulos, enquanto as demais colunas não possuem valores ausentes.

Como a proporção de valores ausentes é relativamente baixa e essas colunas podem fornecer informações relevantes para análises específicas, foi decidido manter os registros na base, pois aremoção dessas linhas reduziria o número de observações disponíveis e poderia eliminar informações importantes de outras variáveis.

---
## Padronização de textos

Remoção de espaços extras no conteúdo das colunas textuais.

In [136]:
for column in text_columns:
    clean_imdb_df[column] = (
        clean_imdb_df[column]
        .str.strip()
    )

---
## Criação e remoção de colunas

Para otimizar o processo de análise dos dados, colunas que não acrescentam a esse processo, como a coluna `poster_link`, serão removidas. Além disso, serão criadas novas variáveis para facilitar a análise.

### Remoção da coluna `poster_link`

In [137]:
clean_imdb_df = clean_imdb_df.drop('poster_link', axis=1)
clean_imdb_df

,series_title,released_year,certificate,runtime,genre,imdb_rating,overview,meta_score,director,star1,star2,star3,star4,no_of_votes,gross
0,The Shawshank Redemption,1994,NC-17,142,[Drama],9.3,Two imprisoned men bond over a number of years...,80.0,Frank Darabont,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler,2343110,28341469.0
1,The Godfather,1972,NC-17,175,"[Crime, Drama]",9.2,An organized crime dynasty's aging patriarch t...,100.0,Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,Diane Keaton,1620367,134966411.0
2,The Dark Knight,2008,PG-13,152,"[Action, Crime, Drama]",9.0,When the menace known as the Joker wreaks havo...,84.0,Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine,2303232,534858444.0
3,The Godfather: Part II,1974,NC-17,202,"[Crime, Drama]",9.0,The early life and career of Vito Corleone in ...,90.0,Francis Ford Coppola,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton,1129952,57300000.0
4,12 Angry Men,1957,G,96,"[Crime, Drama]",9.0,A jury holdout attempts to prevent a miscarria...,96.0,Sidney Lumet,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler,689845,4360000.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,Breakfast at Tiffany's,1961,NC-17,115,"[Comedy, Drama, Romance]",7.6,A young New York socialite becomes interested ...,76.0,Blake Edwards,Audrey Hepburn,George Peppard,Patricia Neal,Buddy Ebsen,166544,NaN
996,Giant,1956,G,201,"[Drama, Western]",7.6,Sprawling epic covering the life of a Texas ca...,84.0,George Stevens,Elizabeth Taylor,Rock Hudson,James Dean,Carroll Baker,34075,NaN
997,From Here to Eternity,1953,G,118,"[Drama, Romance, War]",7.6,"In Hawaii in 1941, a private is cruelly punish...",85.0,Fred Zinnemann,Burt Lancaster,Montgomery Clift,Deborah Kerr,Donna Reed,43374,30500000.0
998,Lifeboat,1944,NaN,97,"[Drama, War]",7.6,Several survivors of a torpedoed merchant ship...,78.0,Alfred Hitchcock,Tallulah Bankhead,John Hodiak,Walter Slezak,William Bendix,26471,NaN


### Criação da coluna `decade` (década de lançamento do filme)

In [138]:
clean_imdb_df["decade"] = (
    ((clean_imdb_df["released_year"] // 10 * 10).astype(str) + 's')
    .astype("category")
)

clean_imdb_df["decade"].value_counts()


decade
2010s    242
2000s    237
1990s    151
1980s     89
1970s     76
1960s     73
1950s     56
1940s     35
1930s     24
1920s     11
2020s      6
Name: count, dtype: int64

### Criação da coluna `genre_count` (contagem de gêneros do filme)

In [139]:
clean_imdb_df["genre_count"] = (
    clean_imdb_df["genre"].str.len()
)

### Criação da coluna `primary_genre` (gênero principal do filme)

In [140]:
clean_imdb_df["primary_genre"] = (
    clean_imdb_df["genre"].str[0]
)
clean_imdb_df["primary_genre"].value_counts()

primary_genre
Drama        289
Action       172
Comedy       155
Crime        107
Biography     88
Animation     82
Adventure     72
Mystery       12
Horror        11
Western        4
Film-Noir      3
Fantasy        2
Family         2
Thriller       1
Name: count, dtype: int64

---
## Conclusão

As principais transformações realizadas foram:

  - padronização dos nomes das colunas para o padrão snake_case
  - conversão dos tipos de dados para formatos mais adequados
  - correção de inconsistências nas colunas `certificate e released_year`
  - padronização das colunas textuais
  - remoção da coluna `poster_link`, por não ser relevante para os objetivos da análise
  - criação das colunas `decade, genre_count e primary_genre`

Os valores ausentes foram mantidos nas colunas `certificate, meta_score e gross`, pois representam uma pequena parcela da base e serão tratados apenas quando necessário em análises específicas.

Com essas transformações, a base está pronta para a etapa de análise exploratória dos dados.

In [141]:
clean_imdb_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 18 columns):
 #   Column         Non-Null Count  Dtype   
---  ------         --------------  -----   
 0   series_title   1000 non-null   string  
 1   released_year  1000 non-null   int64   
 2   certificate    899 non-null    category
 3   runtime        1000 non-null   int64   
 4   genre          1000 non-null   object  
 5   imdb_rating    1000 non-null   float64 
 6   overview       1000 non-null   string  
 7   meta_score     843 non-null    float64 
 8   director       1000 non-null   string  
 9   star1          1000 non-null   string  
 10  star2          1000 non-null   string  
 11  star3          1000 non-null   string  
 12  star4          1000 non-null   string  
 13  no_of_votes    1000 non-null   int64   
 14  gross          831 non-null    float64 
 15  decade         1000 non-null   category
 16  genre_count    1000 non-null   int64   
 17  primary_genre  1000 non-null   obj

---

## Exportação da base após limpeza

In [142]:
clean_imdb_df.to_csv(
    "../data/processed/imdb_clean.csv",
    index=False
)

In [143]:
processed_imdb_df = pd.read_csv("../data/processed/imdb_clean.csv")

processed_imdb_df.head()

,series_title,released_year,certificate,runtime,genre,imdb_rating,overview,meta_score,director,star1,star2,star3,star4,no_of_votes,gross,decade,genre_count,primary_genre
0,The Shawshank Redemption,1994,NC-17,142,['Drama'],9.3,Two imprisoned men bond over a number of years...,80.0,Frank Darabont,Tim Robbins,Morgan Freeman,Bob Gunton,William Sadler,2343110,28341469.0,1990s,1,Drama
1,The Godfather,1972,NC-17,175,"['Crime', 'Drama']",9.2,An organized crime dynasty's aging patriarch t...,100.0,Francis Ford Coppola,Marlon Brando,Al Pacino,James Caan,Diane Keaton,1620367,134966411.0,1970s,2,Crime
2,The Dark Knight,2008,PG-13,152,"['Action', 'Crime', 'Drama']",9.0,When the menace known as the Joker wreaks havo...,84.0,Christopher Nolan,Christian Bale,Heath Ledger,Aaron Eckhart,Michael Caine,2303232,534858444.0,2000s,3,Action
3,The Godfather: Part II,1974,NC-17,202,"['Crime', 'Drama']",9.0,The early life and career of Vito Corleone in ...,90.0,Francis Ford Coppola,Al Pacino,Robert De Niro,Robert Duvall,Diane Keaton,1129952,57300000.0,1970s,2,Crime
4,12 Angry Men,1957,G,96,"['Crime', 'Drama']",9.0,A jury holdout attempts to prevent a miscarria...,96.0,Sidney Lumet,Henry Fonda,Lee J. Cobb,Martin Balsam,John Fiedler,689845,4360000.0,1950s,2,Crime
